In [1]:
import pandas as pd
from datasets import Dataset, DatasetDict

df = pd.read_json('data.json', lines=True)
df["instruction"]="Segment the following feedback into a summary."

/Users/sathishkumarchandran/IdeaProjects/llm/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
 #Convert to Hugging Face Dataset format
dataset = Dataset.from_pandas(df)

# Split the small dataset into train and test
dataset = dataset.train_test_split(test_size=0.2)
# For very small datasets like this, you might not even need a test set
# or just train on the whole thing and evaluate manually.
# For simplicity, we will split it.

# The dataset dictionary will contain 'train' and 'test' splits
# Create a DatasetDict for easier use with Trainer
dataset_dict = DatasetDict({
    'train': dataset['train'],
    'test': dataset['test']
})

print(dataset_dict)


DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2000
    })
})


In [3]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

model_name = "t5-small"  # A smaller, faster model for demonstration
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


In [4]:
def preprocess_function(examples):
    inputs = [f"{examples['instruction']} {examples['input']}" for _ in examples['input']]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    labels = tokenizer(examples['output'], max_length=150, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_dict.map(preprocess_function, batched=True)


Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [01:08<00:00, 29.39 examples/s]


In [5]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})

In [ ]:
import numpy as np
import evaluate
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# --- Step 5: Fine-tune the model ---
# For fine-tuning a sequence-to-sequence model like T5,
# use Seq2SeqTrainingArguments and Seq2SeqTrainer.

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-summarization-finetuned",  # Directory to save the model checkpoints
    eval_strategy="epoch",                # Evaluate after each epoch
    learning_rate=2e-5,
    per_device_train_batch_size=4,              # Adjust based on your GPU memory
    per_device_eval_batch_size=4,               # Adjust based on your GPU memory
    weight_decay=0.01,
    save_total_limit=3,                         # Keep only the last 3 checkpoints
    num_train_epochs=3,                         # Small number of epochs for demonstration
    fp16=True,                                  # Use mixed precision for faster training
    predict_with_generate=True,                 # Generate predictions for evaluation
    logging_steps=10,
    report_to="none"                            # Disable logging to services like Weights & Biases
)

# Load the ROUGE metric
rouge = evaluate.load("rouge")

# Define the compute_metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in the labels as they were not processed by the model
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # ROUGE metric expects a list of predictions and a list of references
    # It also handles different types of ROUGE scores (1, 2, L)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    
    # Calculate the average length of predictions
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

# Define the data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Initialize the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],    
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()





/var/folders/cf/1n82cp8j5477wf_t828mwrlh0000gn/T/ipykernel_2102/2946150843.py:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
W1028 10:12:57.654000 2102 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
/Users/sathishkumarchandran/IdeaProjects/llm/venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


In [ ]:
# --- Step 6: Use the fine-tuned model for inference ---
from transformers import pipeline

# The trainer saves the model at the specified output directory
finetuned_model_path = "./t5-summarization-finetuned/checkpoint-3"
summarizer = pipeline("summarization", model=finetuned_model_path, tokenizer=tokenizer)

# Use the trained model to generate a new summary
new_feedback = "The project was consistently over budget and missed several key deadlines. The team, however, was highly diverse and collaborated effectively."
prompt = f"Instruction: Segment the following feedback into a summary. Input: {new_feedback}"

# Generate the summary
summary = summarizer(prompt, max_length=50, min_length=10, do_sample=False)

print(f"Original: {new_feedback}")
print(f"Generated Summary: {summary[0]['summary_text']}")

In [ ]:
# !pip install evaluate
# !pip install nltk absl-py rouge-score


In [ ]:
from transformers import pipeline

finetuned_model_path = trainer.state.best_model_checkpoint if trainer.state.best_model_checkpoint else "./t5-summarization-finetuned"
summarizer = pipeline("summarization", model=finetuned_model_path, tokenizer=tokenizer)

new_feedback = "The software update introduced several bugs that caused the application to crash frequently, but the new user interface is a major improvement."
prompt = f"Instruction: Segment the following feedback into a summary. Input: {new_feedback}"

summary = summarizer(prompt, max_length=50, min_length=10, do_sample=False)

print(f"Original: {new_feedback}")
print(f"Generated Summary: {summary[0]['summary_text']}")
